# Instructions

1. Assign subject to a group using [this random number generator](https://www.calculator.net/random-number-generator.html?slower=1&supper=4&ctype=1&s=5113&x=Generate).
2. Enter the date, subject ID, and group info into [the table on the timeline slides](https://docs.google.com/presentation/d/1vPCzzxUqa3gjfo_nY4hteUsnp4uf-nPtYLgrwr6Yat8/edit#slide=id.g2fc55de8142_0_754)
3. Show the participant the [task explanation slides](https://docs.google.com/presentation/d/1gwbIJBaGXZgyK8oOKZCZh0Squ2SrUesE2eKkt39Vv1I/edit#slide=id.p). Read directly off of the slides -- try not to add any additional instructions or explanations.
4. Have the participant sign the [consent form](https://redcap.iths.org/surveys/?s=FCXJ4HP7KEM3EJWW).
5. Make sure that chair legs are overlying tape markers on the floor, and seat the participant.
6. On the BMI3D computer, double check that the “master” branch of BMI3D is checked out in VScode (bottom left corner) and click the little refresh symbol to ensure you have the most recent version.
7. In Google Chrome (BMI3D computer), click the BMI3d bookmark in the bookmarks bar and click Setup. Click Subjects and add a new subject with the subject ID. Subject name should be the study abbreviation followed by the subject’s assigned number, i.e. CES006
8. Measure from the floor to the participants’ eye level (in cm)
9. Put your first name (capitalized) as the experiment, the subject number, group number, and the participant's eye height in the variables below.

In [ ]:
experimenter = "Victoria"
subject = "test"
group = 2
eye_height = 130 # measure in cm

save = False # save data (false is test)




#### ---- Do not edit below here --- ####

if not experimenter[0].isupper():
    print("Warning: Capitalize your name for consistency. Check spelling.")
if subject[:3] != "CES" or not subject[3:].isdigit() or len(subject) != 6:
    print(subject[:3] != "CES", ~subject[3:].isdigit(), len(subject) != 6)
    print("Warning: subject names must be CES followed by three-digit number.")
if group not in [1,2,3,4]:
    print("Warning: experimental condition not valid. Choose from:\n"
    "1: Congruent Upright Display/Upright Mapping, \n"
    "2: Incongruent Upright Display/Flat Mapping, \n"
    "3: Congruent Flat Display/Flat Mapping, \n"
    "4: Incongruent Flat Display/Upright Mapping"
    )

10. Run the **Default Settings** cell. You can easily run each cell by holding *shift* and pressing *return* on the keyboard.  
11. Run each block one by one, in order, by running the corresponding cell. A "completed" message will print and a bell sound will play when the block has completed. Give a few minutes of break between each block before prompting the participant on whether they are ready to continue, and running the next cell.

# Default settings

In [ ]:
# Generate default settings

from riglib.experiment.scripting import run_experiment
from datetime import datetime
import os

os.environ['BMI3D_PORT'] = "8000"

project = "CES"
task = "manual control"

features = ["autostart",
     "optitrack",
     "penalty_audio",
     "rand_delay",
     "reward_audio",
     "saveHDF",
     "syncHDF",
     "windowVR"]

parameters = {
    "scale": 1,
    "offset": [-20, -95, -2],
    "reward_sound": "incorrect.wav",
    "penalty_sounds": "buzzer.wav",
    "rand_start": [0.25, 0.75],
    "rand_delay": [0,0],
    "prob_catch_trials": 0,
    "grid_size": 130,
    "fixed_camera_position": False,
    "fixed_camera_orientation": False,
    "fullscreen": True,
    "session_length": 0,
    "velocity_control": False,
    "random_rewards": False,
    "rotation": "yzx",
    "exp_rotation": "none",
    "exp_gain": 1.0,
    "limit2d": False,
    "target_radius": 1,
    "plant_type": "cursor",
    "plant_visible": True,
    "reward_time": 0.5,
    "hold_time": 0.2,
    "hold_penalty_time": 1,
    "delay_penalty_time": 1,
    "timeout_time": 10,
    "timeout_penalty_time": 1,
    "max_attempts": 3,
    "num_targets_per_attempt": 2
    }

if group==1 or group==2: # Upright display
    generator = "centerout_2d"
    parameters['cursor_bounds'] = [-14,14,0,0,-14,14]
    parameters['grid_position'] = [0, 0, 130-eye_height]
    parameters['camera_offset'] = [0, -eye_height, 50]
elif group==3 or group==4: # Flat display
    generator = "centerout_tabletop"
    parameters['cursor_bounds'] = [-14,14,-14,14,0,0]
    parameters['grid_position'] = [0, 0, 160-eye_height]
    parameters['camera_offset'] = [0, 30-eye_height, 50]

settings = {"features": features, "parameters": parameters}

blocks_remaining = 12

import pickle
with open('CES_defaults.pkl', 'wb') as f:
    pickle.dump(settings, f)

from IPython.display import HTML
def play_sound(file):
    return HTML(f"""
    <audio autoplay>
      <source src="{file}" type="audio/mpeg">
    </audio>
    """)

# Blind Block (z1)
* Remind participants that they will not see a cursor for this block, but to move their hand from the center to where they think the target is and hold there until the target disappears.
* Let them know that this is not a calibration, so what they do here will not have any effect on how the task works later.

In [ ]:
import pickle
with open('CES_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']

session = "z1"
nblocks = 4

task_desc = "Zero-feedback"

remove_feats = ['penalty_audio','reward_audio']
features = [feat for feat in features if feat not in remove_feats]

parameters['exp_gain'] = 0
parameters['plant_visible'] = False
parameters['timeout_time'] = 1.5
parameters['hold_time'] = 1.5
parameters['hold_penalty_time'] = 0
parameters['delay_penalty_time'] = 0
parameters['timeout_penalty_time'] = 0
parameters['rand_start'] = [0.25, 0.75]
parameters['max_attempts'] = 1

sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'Blind block (z1) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

# Baseline Session
* Let them know that they'll be able to see the cursor now.
* Let the incongruent group know beforehand: "This task may be more difficult than how it sounded. I promise that it's possible to do and that it's working as intended. Just stick in there."
* Don't give further details on the control mapping. If they ask, just say: "That's not something that I can tell you."

In [ ]:
# Block 1 (a1)

with open('CES_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']
    
session = "a1"
nblocks = 8

if group==1 or group==3: # congruent
    parameters['exp_rotation'] = "none"
    task_desc = "Congruent baseline"
elif group==2 or group==4: # incongruent
    parameters['exp_rotation'] = "about_x_90"
    task_desc = "Incongruent baseline"
    
sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'First baseline block (a1) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

In [ ]:
# Block 2 (a2)

session = "a2"

run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'Second baseline block (a2) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

In [ ]:
# Block 3 (a3)

session = "a3"

run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'Third baseline block (a3) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

# Catch Block (z2)
* "You will once again not be able to see the cursor, but now that you have some experience with the cursor, the goal is to try to bring your hand to where you think the cursor would be in the target."
* Again, this isn't a calibration so it isn't going to be used to change cursor behavior in the future.

In [ ]:
with open('CES_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']

session = "z2"
nblocks = 4

task_desc = "Zero-feedback"

remove_feats = ['penalty_audio','reward_audio']
features = [feat for feat in features if feat not in remove_feats]

parameters['exp_gain'] = 0
parameters['plant_visible'] = False
parameters['timeout_time'] = 1.5
parameters['hold_time'] = 1.5
parameters['hold_penalty_time'] = 0
parameters['delay_penalty_time'] = 0
parameters['timeout_penalty_time'] = 0
parameters['rand_start'] = [0.25, 0.75]
parameters['max_attempts'] = 1

sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'Catch block (z2) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

# Perturbation Session
* Let them know they'll be controlling the cursor again. Don't say anything about the mapping, including confirming to them that it's changed. If asked, say "That's not something that I can tell you."

In [ ]:
# Block 1 (b1)

with open('CES_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']
    
session = "b1"
nblocks = 6

if group==1 or group==2: # Upright display
    task_desc = "Perturbation (upright_equal45)"
    parameters['exp_rotation'] = "upright_equal45"
elif group==3 or group==4: # Flat display
    task_desc = "Perturbation (flat_equal45)"
    parameters['exp_rotation'] = "flat_equal45"
    
sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'First perturbation block (b1) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

In [ ]:
# Block 2 (b2)
    
session = "b2"
    
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'Second perturbation block (b2) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

In [ ]:
# Block 3 (b3)
    
session = "b3"
    
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'Third perturbation block (b3) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

In [ ]:
# Block 4 (b4)
    
session = "b4"
    
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'Fourth perturbation block (b4) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

# Catch Block (z3)
* Cursor is going to be hidden again.
* Reiterate that the goal is to put your hand where you think it would bring the cursor into the target.

In [ ]:
with open('CES_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']

session = "z3"
nblocks = 4

task_desc = "Zero-feedback"

remove_feats = ['penalty_audio','reward_audio']
features = [feat for feat in features if feat not in remove_feats]

parameters['exp_gain'] = 0
parameters['plant_visible'] = False
parameters['timeout_time'] = 1.5
parameters['hold_time'] = 1.5
parameters['hold_penalty_time'] = 0
parameters['delay_penalty_time'] = 0
parameters['timeout_penalty_time'] = 0
parameters['rand_start'] = [0.25, 0.75]
parameters['max_attempts'] = 1

sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'Catch block (z3) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

# Washout Session
* Don't tell them anything will change between the last block and this one. Say "That's not something I can tell you" if they ask if it changed, or stay silent if they make a comment.

In [ ]:
# Block 1 (aprime1)

with open('CES_defaults.pkl', 'rb') as f:
    settings = pickle.load(f)
    
parameters = settings['parameters']
features = settings['features']
    
session = "aprime1"
nblocks = 8

if group==1 or group==3: # congruent
    parameters['exp_rotation'] = "none"
    task_desc = "Congruent washout"
elif group==2 or group==4: # incongruent
    parameters['exp_rotation'] = "about_x_90"
    task_desc = "Incongruent washout"
    
sequence = generator + ":[nblocks=" + str(nblocks) + ", ntargets=8, distance=10, origin=0 0 0]"
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()
blocks_remaining = blocks_remaining - 1

print(f'First washout block (aprime1) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.\n Blocks remaining: {blocks_remaining}.')
play_sound("Bell.wav")

In [ ]:
# Block 2 (aprime2)

session = "aprime2"
    
run_experiment(subject,experimenter,project,session,task,features,sequence,task_desc,save,**parameters)

now = datetime.now()

print(f'Second washout block (aprime2) completed at {now.strftime("%I:%M %p, %m-%d-%Y")}.')
play_sound("Bell.wav")

# print("Don't forget to put files on moor!")